# Time series cross validation: a gap changes the question

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/time_series_cross_validation.ipynb)

A self-contained CPU notebook. All data are generated locally; no credentials or repository files are needed.

In [ ]:
%pip -q install numpy scipy scikit-learn matplotlib

## Experiment design

The 12-step target is the mean of the next 12 observations. Each input contains the current observation and 23 previous ones. Development targets end strictly before the first future forecast origin. The future evaluation refits once and predicts each origin using observations available at that origin; it does not refit between predictions. Fold-level scaling is fitted only on training rows.

The gap addresses target availability. A later training origin may have a label extending beyond the validation origin even when input rows are ordered. A regime change makes early folds harder than the final future segment, so preserving chronology need not produce an error estimate numerically closer to future error.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import math


In [ ]:
def forecast_data(seed=0):
    rng=np.random.default_rng(seed)
    n=1800; z=np.zeros(n)
    for t in range(1,n):z[t]=.82*z[t-1]+rng.normal(0,.5)
    t=np.arange(n)
    series=2*np.sin(2*np.pi*t/48)+z+np.where(t>=900,3.,0.)
    origins=np.arange(24,n-12)
    X=np.array([series[o-np.arange(24)] for o in origins])
    y=np.array([series[o+1:o+13].mean() for o in origins])
    return series,origins,X,y


In [ ]:
def forecast_experiment():
    records=[]; example=None
    for seed in range(5):
        series,origins,X,y=forecast_data(seed)
        # Every development target is available before the first future origin.
        dev=np.flatnonzero(origins+12<1400); future=np.flatnonzero(origins>=1400)
        methods={'Shuffled':KFold(5,shuffle=True,random_state=42),
                 'Expanding':TimeSeriesSplit(5),
                 'Gap 12':TimeSeriesSplit(5,gap=12)}
        for name,splitter in methods.items():
            scores=[]
            for alpha in [.01,1.,100.,10000.]:
                errors=[]
                for tr,va in splitter.split(dev):
                    assert not np.intersect1d(tr, va).size
                    if name == 'Gap 12':
                        assert origins[dev[tr[-1]]] + 12 < origins[dev[va[0]]]
                    model=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
                    model.fit(X[dev[tr]],y[dev[tr]])
                    errors.append(mean_squared_error(y[dev[va]],model.predict(X[dev[va]])))
                scores.append(np.mean(errors))
            selected=int(np.argmin(scores)); alpha=[.01,1.,100.,10000.][selected]
            model=make_pipeline(StandardScaler(),Ridge(alpha=alpha)).fit(X[dev],y[dev])
            pred=model.predict(X[future]); rmse=mean_squared_error(y[future],pred)**.5
            records.append({'seed':seed,'method':name,'alpha':alpha,'cv_rmse':float(scores[selected]**.5),'future_rmse':float(rmse)})
            if seed==0 and name=='Gap 12': example=(series,origins,X,y,dev,future,pred)
    return records,example


In [ ]:
records, example = forecast_experiment()
for name in ["Shuffled", "Expanding", "Gap 12"]:
    rows = [r for r in records if r["method"] == name]
    print(name, "CV RMSE", np.mean([r["cv_rmse"] for r in rows]),
          "future RMSE", np.mean([r["future_rmse"] for r in rows]),
          "selected alphas", [r["alpha"] for r in rows])
series, origins, X, y, dev, future, pred = example
assert np.max(origins[dev]+12) < np.min(origins[future])
plt.figure(figsize=(9,3)); plt.plot(series); plt.axvline(1400,color="black",linestyle="--")
plt.xlabel("Time index"); plt.ylabel("Value"); plt.title("Synthetic series and future holdout"); plt.show()


## Expanding folds and available targets

The animation uses the same five TimeSeriesSplit folds as the experiment. Each fold has three frames: its training origins, the last training target, and the first validation forecast. The grey interval excludes twelve forecast origins. Observations in that interval still complete earlier training labels. Each final training target ends strictly before the first validation origin. The second panel enlarges that boundary so the twelve-step gap remains visible.

In [ ]:
def forecast_animation(origins, horizon=12):
    """Animate actual expanding folds and the availability of their last label."""
    from matplotlib.animation import FuncAnimation

    origins = np.asarray(origins)
    splits = list(TimeSeriesSplit(n_splits=5, gap=horizon).split(origins))
    checks = []
    for train, valid in splits:
        last_origin = int(origins[train[-1]])
        first_valid = int(origins[valid[0]])
        label_end = last_origin + horizon
        assert label_end < first_valid
        checks.append({"last_training_origin": last_origin,
                       "last_training_label_end": label_end,
                       "first_validation_origin": first_valid,
                       "excluded_origins": first_valid - last_origin - 1})

    fig, (overview, detail) = plt.subplots(
        2, 1, figsize=(10, 6), gridspec_kw={"height_ratios": [1.2, 1]})
    teal, amber, blue = "#167d8d", "#d88a19", "#526b98"
    phases = ["Training origins", "Last training target", "First validation forecast"]

    def update(frame):
        fold, phase = divmod(frame, 3)
        overview.clear()
        detail.clear()
        for i, (train, valid) in enumerate(splits):
            opacity = 1 if i == fold else .18
            overview.plot(origins[train], np.full(len(train), i),
                          color=teal, linewidth=6, alpha=opacity)
            overview.plot(origins[valid], np.full(len(valid), i),
                          color=amber, linewidth=6, alpha=opacity)
        overview.set(xlim=(origins[0]-10, origins[-1]+10), ylim=(-.5, 4.5),
                     yticks=range(5), yticklabels=range(1, 6),
                     xlabel="Forecast origin", ylabel="Expanding fold")
        overview.set_title(f"Fold {fold+1} of 5: {phases[phase]}")
        row = checks[fold]
        last, end, valid = (row["last_training_origin"],
                            row["last_training_label_end"],
                            row["first_validation_origin"])
        # The grey interval excludes origins, not the observations needed
        # to complete an earlier training target.
        detail.axvspan(last+.5, valid-.5, color="#dddddd", alpha=.55)
        detail.scatter([last], [1], color=teal, s=85,
                       alpha=1 if phase == 0 else .55, zorder=3)
        detail.plot([last+1, end], [1, 1], color=teal, linewidth=7,
                    alpha=1 if phase == 1 else .4, solid_capstyle="butt")
        detail.scatter([valid], [0], color=amber, s=85,
                       alpha=1 if phase == 2 else .4, zorder=3)
        detail.plot([valid+1, valid+horizon], [0, 0], color=amber,
                    linewidth=7, alpha=1 if phase == 2 else .25,
                    solid_capstyle="butt")
        detail.axvline(valid, color=blue, linestyle="--", linewidth=1)
        detail.text(last, 1.3, f"origin {last}", ha="center", fontsize=9)
        detail.text(end, .7, f"label ends {end}", ha="right", fontsize=9)
        detail.text(valid, -.3, f"validation begins {valid}", ha="left", fontsize=9)
        detail.text(last+1, 1.65, f"{horizon} excluded origins", fontsize=9)
        detail.set(xlim=(last-3, valid+horizon+3), ylim=(-.65, 1.9),
                   yticks=[0, 1], yticklabels=["Validation", "Last training row"],
                   xlabel="Time index near the boundary")
        detail.set_title(f"Target available before validation: {end} < {valid}", fontsize=11)
        fig.tight_layout()
        return []

    animation = FuncAnimation(fig, update, frames=15, interval=1000/3,
                              repeat=True, blit=False)
    return animation, checks

# Use actual development origins from the experiment above.
# Pillow writes the GIF without an external video encoder.
from matplotlib.animation import PillowWriter
from IPython.display import Image, display
animation, fold_checks = forecast_animation(origins[dev], horizon=12)
animation.save('forecast_folds.gif', writer=PillowWriter(fps=3))
plt.close(animation._fig)
display(Image(filename='forecast_folds.gif'))
print(fold_checks)


## Exercises

1. Change the horizon to six throughout the generator, development boundary and gap. Check label endpoints explicitly.
2. Replace the expanding window with a fixed-length rolling window. Select its length using development data only.
3. Remove the level shift and repeat all five seeds. Keep the final future block untouched until selection is finished.